In [10]:
class Car():
    def __init__(self, model, year, hp, fuel_capacity_l):
        self.model = model
        self.year = year
        self.hp = hp
        self.fuel_capacity_l = fuel_capacity_l
        self.fuel_capacity_g = fuel_capacity_l * 0.26

    def display_car_info(self):
        # Display the information of the car in a nice way
        info = f"""
        =============================
        ========- CAR INFO -=========
        = Model     : {self.model}
        = Year      : {self.year}
        = HP        : {self.hp}
        = FuelC / L : {self.fuel_capacity_l} L
        = FuelC / G : {self.fuel_capacity_g} g
        =============================
        =============================
        """
        print(info)


In [11]:
car_1 = Car(model = "BMW",
            year = 2026,
            hp=200,
            fuel_capacity_l=60)

car_2 = Car(model = "Toyota", year = 2025, hp = 120, fuel_capacity_l = 50)

In [12]:
car_1.display_car_info()


        ========- CAR INFO -=========
        = Model     : BMW
        = Year      : 2026
        = HP        : 200
        = FuelC / L : 60 L
        = FuelC / G : 15.600000000000001 g
        


In [13]:
car_2.display_car_info()


        ========- CAR INFO -=========
        = Model     : Toyota
        = Year      : 2025
        = HP        : 120
        = FuelC / L : 50 L
        = FuelC / G : 13.0 g
        


In [14]:
class Item():
    def __init__(self, icon, type_of_item, max_stack):
        self.icon = icon
        self.type_of_item = type_of_item
        self.max_stack = max_stack

    def drop(self):
        # Drop object infront of user
        # Subtract 1 object from players inventory
        pass
        
    def drop_stack(self):
        # Drop stack of items
        # Subtract stack of items from players inventory
        pass
        
    def place(self, x, y, z):
        # place object at x,y,z
        # subtract 1 obj from players inventory
        pass

In [15]:
class Snake():
    def __init__(self, direction, speed):
        self.length = 1
        self.direction = direction
        self.speed = speed
        self.x = 0
        self.y = 0

    def change_direction(self, degree):
        self.direction = self.direction + degree
        pass
    
    def eat_apple(self):
        self.length = self.length+1
        pass

    def die(self):
        # Terminate the game
        pass
        
    def move_one_step(self):
        # Change coordinates to move one step in self.direction
        # If you run into wall or your own tail then self.die
        # If you run into an apple then call self.eat_apple
        # If you don't run into anything, call self.move_one_step
        pass

In [ ]:
import pygame
import random

# Initialize Pygame
pygame.init()

# Constants
SCREEN_WIDTH = 600
SCREEN_HEIGHT = 400
CELL_SIZE = 20
FPS = 10

# Colors (RGB)
BLACK = (0, 0, 0)
WHITE = (200, 200, 200)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
DARK_GREEN = (0, 150, 0)

# Directions as vectors
UP = (0, -1)
DOWN = (0, 1)
LEFT = (-1, 0)
RIGHT = (1, 0)


class Snake:
    """Represents the snake: body segments, direction, and movement."""
    def __init__(self):
        # Start in the middle of the screen, moving right
        start_x = (SCREEN_WIDTH // CELL_SIZE) // 2
        start_y = (SCREEN_HEIGHT // CELL_SIZE) // 2
        self.body = [
            (start_x, start_y),
            (start_x - 1, start_y),
            (start_x - 2, start_y)
        ]
        self.direction = RIGHT
        self.grow_flag = False   # Set to True when food is eaten

    def move(self):
        """Advance the snake by one cell in the current direction."""
        head_x, head_y = self.body[0]
        dx, dy = self.direction
        new_head = (head_x + dx, head_y + dy)

        # Insert new head
        self.body.insert(0, new_head)

        # Remove tail unless we need to grow
        if not self.grow_flag:
            self.body.pop()
        else:
            self.grow_flag = False

    def change_direction(self, direction):
        """Change direction, but prevent reversing into itself."""
        # Prevent reverse direction (e.g., cannot go LEFT if currently RIGHT)
        if (direction[0] * -1, direction[1] * -1) != self.direction:
            self.direction = direction

    def check_self_collision(self):
        """Return True if the head collides with the body."""
        head = self.body[0]
        return head in self.body[1:]

    def check_wall_collision(self):
        """Return True if the head goes outside the grid."""
        head_x, head_y = self.body[0]
        if head_x < 0 or head_x >= SCREEN_WIDTH // CELL_SIZE:
            return True
        if head_y < 0 or head_y >= SCREEN_HEIGHT // CELL_SIZE:
            return True
        return False

    def draw(self, surface):
        """Draw the snake on the given surface."""
        for i, (x, y) in enumerate(self.body):
            rect = pygame.Rect(
                x * CELL_SIZE, y * CELL_SIZE, CELL_SIZE, CELL_SIZE
            )
            # Head is a slightly different shade
            color = DARK_GREEN if i == 0 else GREEN
            pygame.draw.rect(surface, color, rect)
            pygame.draw.rect(surface, BLACK, rect, 1)  # border


class Food:
    """Represents a single food item that the snake can eat."""
    def __init__(self, snake_body):
        self.position = (0, 0)
        self.respawn(snake_body)

    def respawn(self, snake_body):
        """Place food in a random cell not occupied by the snake."""
        max_x = (SCREEN_WIDTH // CELL_SIZE) - 1
        max_y = (SCREEN_HEIGHT // CELL_SIZE) - 1
        while True:
            x = random.randint(0, max_x)
            y = random.randint(0, max_y)
            if (x, y) not in snake_body:
                self.position = (x, y)
                break

    def draw(self, surface):
        """Draw the food as a red square."""
        x, y = self.position
        rect = pygame.Rect(x * CELL_SIZE, y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
        pygame.draw.rect(surface, RED, rect)
        pygame.draw.rect(surface, BLACK, rect, 1)


class Game:
    """Main game class: manages loop, events, and game state."""
    def __init__(self):
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("Snake Game")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Arial", 24)
        self.score = 0
        self.game_over = False

        self.snake = Snake()
        self.food = Food(self.snake.body)

    def handle_events(self):
        """Process user input (keyboard)."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False   # quit the game
            if event.type == pygame.KEYDOWN:
                if self.game_over:
                    # Press any key to restart after game over
                    self.restart()
                else:
                    # Change direction based on arrow keys
                    if event.key == pygame.K_UP:
                        self.snake.change_direction(UP)
                    elif event.key == pygame.K_DOWN:
                        self.snake.change_direction(DOWN)
                    elif event.key == pygame.K_LEFT:
                        self.snake.change_direction(LEFT)
                    elif event.key == pygame.K_RIGHT:
                        self.snake.change_direction(RIGHT)
        return True

    def update(self):
        """Update game logic (move snake, check collisions)."""
        if self.game_over:
            return

        self.snake.move()

        # Check wall collision
        if self.snake.check_wall_collision():
            self.game_over = True
            return

        # Check self collision
        if self.snake.check_self_collision():
            self.game_over = True
            return

        # Check food collision
        if self.snake.body[0] == self.food.position:
            self.snake.grow_flag = True
            self.score += 10
            self.food.respawn(self.snake.body)

    def draw(self):
        """Render everything to the screen."""
        self.screen.fill(BLACK)

        # Draw grid lines (optional, for nicer look)
        for x in range(0, SCREEN_WIDTH, CELL_SIZE):
            pygame.draw.line(self.screen, WHITE, (x, 0), (x, SCREEN_HEIGHT), 1)
        for y in range(0, SCREEN_HEIGHT, CELL_SIZE):
            pygame.draw.line(self.screen, WHITE, (0, y), (SCREEN_WIDTH, y), 1)

        # Draw snake and food
        self.snake.draw(self.screen)
        self.food.draw(self.screen)

        # Draw score
        score_text = self.font.render(f"Score: {self.score}", True, WHITE)
        self.screen.blit(score_text, (10, 10))

        # Game over message
        if self.game_over:
            go_text = self.font.render("GAME OVER - Press any key to restart", True, RED)
            text_rect = go_text.get_rect(center=(SCREEN_WIDTH//2, SCREEN_HEIGHT//2))
            self.screen.blit(go_text, text_rect)

        pygame.display.flip()

    def restart(self):
        """Reset the game to its initial state."""
        self.snake = Snake()
        self.food = Food(self.snake.body)
        self.score = 0
        self.game_over = False

    def run(self):
        """Main game loop."""
        running = True
        while running:
            running = self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(FPS)

        pygame.quit()


if __name__ == "__main__":
    game = Game()
    game.run()